In [1]:
import pandas as pd
import numpy as np
import ast
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")


In [3]:
movies = movies.merge(credits, on="title")


In [4]:
movies = movies[["movie_id","title","genres","overview","keywords","cast","crew"]]
movies.dropna(inplace=True)


In [5]:
# Convert JSON string into list of names
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i["name"])
    return L


In [6]:

# Keep top 3 cast members
def convertUpto3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter < 3:
            L.append(i["name"])
            counter += 1
        else:
            break
    return L


In [7]:

# Extract director
def fetch_director(obj):
    for i in ast.literal_eval(obj):
        if i['job'] == "Director":
            return [i['name']]
    return []


In [ ]:
#cleaning data
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convertUpto3)
movies['crew'] = movies['crew'].apply(fetch_director)


In [ ]:
# removeing spaces inside words for better matching
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

#splitting overview into list of words
movies['overview'] = movies['overview'].apply(lambda x:x.split())



In [10]:
# create a new column 'tags' which contains all the important information about a movie in a single column

movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew'] + movies['overview']

new_df = movies[['movie_id','title','tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


C:\Users\premk\AppData\Local\Temp\ipykernel_18108\1074930108.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [ ]:
# tfidf vectorization 
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(new_df['tags'])


In [ ]:
#computing cosine similarity
similarity = linear_kernel(tfidf_matrix, tfidf_matrix)
similarity.shape


In [19]:
def recommend(movie):
    # Normalize user input (ignore spaces + case)
    movie = movie.strip().lower()
    
    # Normalize titles in DataFrame
    titles_normalized = new_df['title'].str.strip().str.lower()
    
    if movie not in titles_normalized.values:
        print(f"Movie '{movie}' not found in database!")
        return []
    
    movie_index = titles_normalized[titles_normalized == movie].index[0]
    
    distances = similarity[movie_index]
    movie_list = sorted(
        list(enumerate(distances)), 
        reverse=True, key=lambda x: x[1]
    )[1:6]
    
    print(f"\n Top 5 recommendations for '{new_df.iloc[movie_index].title}':")
    recommendations = []
    for i in movie_list:
        rec_title = new_df.iloc[i[0]].title
        print(rec_title)
        recommendations.append(rec_title)
    
    return recommendations



In [22]:
recommend("bee movie")



 Top 5 recommendations for 'Bee Movie':
Honey
Winnie the Pooh
Love Happens
Barry Lyndon
October Baby


['Honey', 'Winnie the Pooh', 'Love Happens', 'Barry Lyndon', 'October Baby']

In [23]:
pickle.dump(movies, open('movies_data.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))
